<a href="https://colab.research.google.com/github/popsql123/skills-sde-2-after-airtel-xlabs/blob/main/audio_analysis_whisper_modified.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎙️ Audio Analysis with OpenAI Whisper
**Transcribe your MP3 audio → Analyse communication & technical quality**

### Steps:
1. Upload your MP3 audio file to Google Drive
2. Transcribe with Whisper
3. Save transcript for analysis
4. Review communication and technical feedback

> 💡 **Tip:** Go to `Runtime → Change runtime type → T4 GPU` for faster transcription

This version is optimized for MP3 uploads instead of huge video files. Because uploading 2GB videos repeatedly is how humans accidentally reinvent suffering.


In [1]:
# ✅ STEP 1: Install required packages
!pip install -q openai-whisper
!apt-get install -qq ffmpeg
print('✅ Installation complete!')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 20.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
✅ Installation complete!


In [2]:
# ✅ STEP 2: Mount Google Drive (to access your video file)
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive mounted!')
print('📁 Your Drive is accessible at: /content/drive/MyDrive/')

Mounted at /content/drive
✅ Google Drive mounted!
📁 Your Drive is accessible at: /content/drive/MyDrive/


In [6]:
# ✅ STEP 3: Set your MP3 audio file path
# Upload your MP3 file to Google Drive first

AUDIO_PATH = '/content/drive/MyDrive/InterviewRecordings/Mobikwik.mp3'  # ← CHANGE THIS

import os

if os.path.exists(AUDIO_PATH):
    size_mb = os.path.getsize(AUDIO_PATH) / (1024**2)
    print(f'✅ Audio file found! Size: {size_mb:.1f} MB')
else:
    print('❌ File not found. Check your Google Drive path.')


✅ Audio file found! Size: 124.4 MB


In [7]:
# ✅ STEP 4: Verify audio file
import os

if os.path.exists(AUDIO_PATH):
    print('🎵 Audio ready for transcription!')
else:
    raise FileNotFoundError('Audio file missing. Fix AUDIO_PATH first.')


🎵 Audio ready for transcription!


In [8]:
# ✅ STEP 5: Transcribe with Whisper
# Model options:
# 'tiny'   = fastest
# 'base'   = fast
# 'small'  = balanced
# 'medium' = high accuracy
# 'large'  = best accuracy but slow

import whisper

MODEL_SIZE = 'medium'

print(f'📥 Loading Whisper {MODEL_SIZE} model...')
model = whisper.load_model(MODEL_SIZE)

print('🎙️ Transcribing audio...')
result = model.transcribe(AUDIO_PATH, verbose=False, language='en')

transcript = result['text']

print('\n✅ TRANSCRIPTION COMPLETE!\n')
print(transcript[:3000])  # preview first 3000 chars

print(f'\n📝 Total transcript length: {len(transcript)} characters')


📥 Loading Whisper medium model...


100%|██████████████████████████████████████| 1.42G/1.42G [00:15<00:00, 101MiB/s]


🎙️ Transcribing audio...


 98%|█████████▊| 319973/325973 [04:52<00:05, 1092.41frames/s]


✅ TRANSCRIPTION COMPLETE!

 Open any IDE Is my screen visible to you? Yes, your screen is visible but you are not Yes, now you are visible Any IDE So like any online IDE or my... You can use your personal IDE I will show you You will see The screen will be visible You can see You can see it You can see the screen You can see the screen You can see the screen So, it will take a minute Actually, it's my personal laptop now I know this So, I need 070 That is actually useful You can use an online IDE I don't think so It will be crashed It will not crash but It will sometimes lag Yeah, definitely Open it Is this opened? Yes, it's opened So Create a pojo Where we have Subject name and subject Marks I need to identify each and every subject Second highest by using stream Subject, student name and marks Subject, marks Subject, student name Okay And I will have to use streams for getting the highest marks Second highest mark Second highest subject wise mark Suppose A having second highest in m

In [ ]:
# ✅ STEP 6: Save transcript to Google Drive
TRANSCRIPT_PATH = '/content/drive/MyDrive/InterviewRecordings/Mobikwik.txt'

with open(TRANSCRIPT_PATH, 'w', encoding='utf-8') as f:
    f.write('=== AUDIO TRANSCRIPT ===\n\n')
    f.write(transcript)

print(f'✅ Transcript saved to Google Drive: {TRANSCRIPT_PATH}')
print('\n📋 Next Step: Paste the transcript into ChatGPT, Claude, or Gemini for analysis.')


In [ ]:
# ✅ STEP 7: Basic Self-Analysis (Filler words, speaking pace)
import re
from collections import Counter

words = transcript.lower().split()
total_words = len(words)

# Filler word detection
fillers = ['um', 'uh', 'like', 'you know', 'basically', 'actually', 'literally',
           'right', 'so', 'kind of', 'sort of', 'i mean', 'okay', 'well']
filler_counts = {f: transcript.lower().count(f) for f in fillers if transcript.lower().count(f) > 0}
filler_counts = dict(sorted(filler_counts.items(), key=lambda x: -x[1]))

# Duration estimate from segments
if result.get('segments'):
    duration_min = result['segments'][-1]['end'] / 60
    wpm = total_words / duration_min
else:
    duration_min = None
    wpm = None

print('=' * 50)
print('📊 COMMUNICATION QUICK STATS')
print('=' * 50)
print(f'📝 Total Words Spoken : {total_words:,}')
if duration_min:
    print(f'⏱️  Video Duration     : {duration_min:.1f} minutes')
    print(f'🏃 Speaking Pace      : {wpm:.0f} words/min ', end='')
    if wpm < 120:
        print('(Too slow — aim for 130-160)')
    elif wpm > 180:
        print('(Too fast — aim for 130-160)')
    else:
        print('(✅ Good pace!)')

print(f'\n🗣️  Filler Words Detected:')
if filler_counts:
    for filler, count in list(filler_counts.items())[:8]:
        bar = '█' * min(count, 30)
        print(f'  "{filler}"  {bar}  ({count}x)')
else:
    print('  ✅ Very few filler words detected!')

print('\n' + '=' * 50)
print('✅ Now paste the full transcript into Claude for deep analysis!')
print('=' * 50)

In [ ]:
# ✅ STEP 8: Generate timestamped segments (useful for reviewing specific moments)
print('⏱️  TIMESTAMPED SEGMENTS (first 20)\n')
print(f'{"Time":<12} {"Text"}')
print('-' * 70)

for seg in result.get('segments', [])[:20]:
    start = seg['start']
    mins = int(start // 60)
    secs = int(start % 60)
    print(f'[{mins:02d}:{secs:02d}]      {seg["text"].strip()}')

# Save timestamped version
TIMESTAMPED_PATH = '/content/drive/MyDrive/InterviewRecordings/Mobikwik.txt'
with open(TIMESTAMPED_PATH, 'w', encoding='utf-8') as f:
    f.write('=== TIMESTAMPED TRANSCRIPT ===\n\n')
    for seg in result.get('segments', []):
        start = seg['start']
        mins = int(start // 60)
        secs = int(start % 60)
        f.write(f'[{mins:02d}:{secs:02d}] {seg["text"].strip()}\n')

print(f'\n✅ Timestamped transcript saved to: {TIMESTAMPED_PATH}')

## 🔍 What to do with your transcript

Now that you have the transcript, paste it into **Claude.ai** with these prompts:

### For Communication Analysis:
```
Analyse this transcript for:
1. Filler words and verbal habits
2. Clarity and conciseness
3. Confidence and tone
4. Structure and flow
5. Top 5 specific improvements I should make
```

### For Technical Content Analysis:
```
Review this technical transcript:
1. Is the technical explanation accurate?
2. Is it well-structured and easy to follow?
3. Are there gaps or missing explanations?
4. What topics need more depth?
5. Rate the overall technical quality out of 10
```

---
> Made for free video analysis using OpenAI Whisper + Google Colab